# Notebook 08 – Backend Implementation with FastAPI

## 1. Overview

This notebook implements the backend API for the **Advance AI Research Assistant** using FastAPI.

The backend connects the previously developed PDF ingestion, text preprocessing, embedding generation, hybrid retrieval, reranking, and LLM-based answer generation components into a reusable application interface.

It provides REST API endpoints that will later be connected to the Next.js frontend.

## 2. Objectives

- Create a modular FastAPI backend.
- Organize the existing RAG components into reusable Python modules.
- Implement PDF upload and processing.
- Integrate ChromaDB for persistent vector storage.
- Connect semantic search, BM25 retrieval, and Cross-Encoder reranking.
- Integrate Ollama for local LLM-based answer generation.
- Implement API endpoints for document upload, question answering, and health checks.
- Return answers with references to the original PDF documents and page numbers.

## 3. Tools and Technologies

| Technology | Purpose |
|---|---|
| Python 3.11 | Backend programming |
| FastAPI | REST API framework |
| Uvicorn | ASGI server |
| Pydantic | Data validation and response models |
| PyMuPDF | PDF text extraction |
| Sentence Transformers | Text embeddings |
| ChromaDB | Persistent vector database |
| BM25 | Keyword-based retrieval |
| Cross-Encoder | Passage reranking |
| Ollama | Local LLM inference |
| Qwen 2.5 (3B) | Answer generation |

## 4. Input and Expected Output

**Input:**
- PDF documents uploaded through the API.
- Research questions submitted by the user.

**Expected output:**
- Extracted and indexed PDF content.
- Retrieved and reranked relevant passages.
- AI-generated answers based on retrieved evidence.
- Source references containing document filenames and page numbers.
- Appropriate error messages and responses when documents or evidence are unavailable.

## 5. Implementation Workflow

1. Inspect the existing project structure.
2. Create the required backend directory structure.
3. Organize reusable components into Python modules.
4. Implement PDF upload and document indexing.
5. Implement the RAG query pipeline.
6. Create FastAPI endpoints.
7. Start the API using Uvicorn.
8. Test the endpoints and verify the complete backend workflow.

## 6. Expected Outcome

At the end of this notebook, the **Advance AI Research Assistant** will have a functional FastAPI backend.

Users will be able to upload scientific PDF documents, submit research questions, and receive AI-generated answers supported by references to the original documents.

The backend will provide the foundation for the Next.js frontend integration.

## Step 1. -Inspect the project structure

In [1]:

from pathlib import Path

# Define the project root
PROJECT_ROOT = (
    Path.home()
    / "Schreibtisch"
    / "IfaDW"
    / "NLP"
    / "Final_Project"
    / "advance-ai-research-assistant"
)

# Verify the project directory
assert PROJECT_ROOT.exists(), f"Project directory not found: {PROJECT_ROOT}"

print("Project root:", PROJECT_ROOT)


# Display the existing project structure
def show_project_tree(directory, max_depth=3, current_depth=0):
    """Display project folders and files up to a specified depth."""

    if current_depth > max_depth:
        return

    for path in sorted(directory.iterdir()):
        # Skip large or unnecessary directories
        if path.name in {
            ".git", ".venv", "__pycache__",
            "node_modules", ".next", ".ipynb_checkpoints"
        }:
            continue

        indent = "    " * current_depth

        if path.is_dir():
            print(f"{indent}[DIR]  {path.name}/")
            if current_depth < max_depth:
                show_project_tree(
                    path,
                    max_depth=max_depth,
                    current_depth=current_depth + 1
                )
        else:
            print(f"{indent}[FILE] {path.name}")


show_project_tree(PROJECT_ROOT, max_depth=3)


Project root: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant
[FILE] .env.example
[FILE] .gitignore
[FILE] SETUP-GUIDE.md
[DIR]  backend/
    [FILE] __init__.py
    [DIR]  app/
        [FILE] __init__.py
    [DIR]  tests/
        [FILE] __init__.py
[DIR]  benchmarks/
[DIR]  data/
    [DIR]  chroma_db/
        [DIR]  68e69dee-38f8-47e3-a4bb-cc43d8c052ea/
            [FILE] data_level0.bin
            [FILE] header.bin
            [FILE] length.bin
            [FILE] link_lists.bin
        [FILE] chroma.sqlite3
    [DIR]  processed/
        [FILE] document_chunks.jsonl
        [FILE] retrieval_test_results.csv
        [FILE] semantic_search_test.csv
    [DIR]  raw_pdfs/
        [FILE] 41598_2019_Article_38798.pdf
        [FILE] NIPS-2017-attention-is-all-you-need-Paper.pdf
        [FILE] s41598-019-52937-5.pdf
    [DIR]  sample_documents/
    [DIR]  uploads/
[DIR]  docs/
[FILE] environment.yml
[DIR]  experiments/
[DIR]  frontend/
[DIR]  logs/
[DIR]  notebooks/


## Step 2 – Inspect the existing backend files

In [2]:

# Inspect existing backend and ingestion files

folders_to_inspect = [
    PROJECT_ROOT / "backend",
    PROJECT_ROOT / "src",
]

for folder in folders_to_inspect:
    print(f"\n{'=' * 65}")
    print(f"FOLDER: {folder.relative_to(PROJECT_ROOT)}")
    print("=" * 65)

    for path in sorted(folder.rglob("*")):
        if any(
            part in {".git", "__pycache__", ".ipynb_checkpoints"}
            for part in path.parts
        ):
            continue

        if path.is_file():
            print(path.relative_to(PROJECT_ROOT))


# Inspect the existing PDF extractor
extractor_file = PROJECT_ROOT / "src" / "ingestion" / "pdf_extractor.py"

if extractor_file.exists():
    print("\n" + "=" * 65)
    print("EXISTING PDF EXTRACTOR")
    print("=" * 65)

    print(extractor_file.read_text(encoding="utf-8"))
else:
    print("\nNo existing PDF extractor found.")



FOLDER: backend

FOLDER: src
src/ingestion/__init__.py
src/ingestion/pdf_extractor.py

EXISTING PDF EXTRACTOR

from pathlib import Path
import fitz


def extract_pdf(pdf_path: str | Path) -> dict:
    """
    Validate a PDF and extract its metadata and page-level text.

    Args:
        pdf_path: Path to the PDF document.

    Returns:
        Dictionary containing document metadata and extracted pages.
    """

    pdf_path = Path(pdf_path)

    # Validate the input file.
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    if pdf_path.suffix.lower() != ".pdf":
        raise ValueError("The file must have a PDF extension.")

    if pdf_path.stat().st_size == 0:
        raise ValueError("The PDF file is empty.")

    pages = []

    try:
        with fitz.open(pdf_path) as document:

            if document.is_encrypted:
                raise ValueError("Password-protected PDFs are not supported.")

            if document.page_count == 0:


## Step 3 – Create the backend package structure

In [3]:

# Create the backend package structure

directories = [
    PROJECT_ROOT / "src",
    PROJECT_ROOT / "src" / "processing",
    PROJECT_ROOT / "src" / "embeddings",
    PROJECT_ROOT / "src" / "retrieval",
    PROJECT_ROOT / "src" / "generation",
    PROJECT_ROOT / "backend" / "app",
    PROJECT_ROOT / "backend" / "tests",
]

# Create directories if they do not already exist
for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)


# Create package initialization files
init_files = [
    PROJECT_ROOT / "src" / "__init__.py",
    PROJECT_ROOT / "src" / "processing" / "__init__.py",
    PROJECT_ROOT / "src" / "embeddings" / "__init__.py",
    PROJECT_ROOT / "src" / "retrieval" / "__init__.py",
    PROJECT_ROOT / "src" / "generation" / "__init__.py",
    PROJECT_ROOT / "backend" / "__init__.py",
    PROJECT_ROOT / "backend" / "app" / "__init__.py",
    PROJECT_ROOT / "backend" / "tests" / "__init__.py",
]

for file_path in init_files:
    file_path.touch(exist_ok=True)

print("Backend package structure created successfully.")

# Verify the new structure
for directory in directories:
    print(f"✓ {directory.relative_to(PROJECT_ROOT)}")


Backend package structure created successfully.
✓ src
✓ src/processing
✓ src/embeddings
✓ src/retrieval
✓ src/generation
✓ backend/app
✓ backend/tests


## Step 4 – Verify the result

In [4]:
show_project_tree(PROJECT_ROOT, max_depth=3)

[FILE] .env.example
[FILE] .gitignore
[FILE] SETUP-GUIDE.md
[DIR]  backend/
    [FILE] __init__.py
    [DIR]  app/
        [FILE] __init__.py
    [DIR]  tests/
        [FILE] __init__.py
[DIR]  benchmarks/
[DIR]  data/
    [DIR]  chroma_db/
        [DIR]  68e69dee-38f8-47e3-a4bb-cc43d8c052ea/
            [FILE] data_level0.bin
            [FILE] header.bin
            [FILE] length.bin
            [FILE] link_lists.bin
        [FILE] chroma.sqlite3
    [DIR]  processed/
        [FILE] document_chunks.jsonl
        [FILE] retrieval_test_results.csv
        [FILE] semantic_search_test.csv
    [DIR]  raw_pdfs/
        [FILE] 41598_2019_Article_38798.pdf
        [FILE] NIPS-2017-attention-is-all-you-need-Paper.pdf
        [FILE] s41598-019-52937-5.pdf
    [DIR]  sample_documents/
    [DIR]  uploads/
[DIR]  docs/
[FILE] environment.yml
[DIR]  experiments/
[DIR]  frontend/
[DIR]  logs/
[DIR]  notebooks/
    [FILE] 00_project_setup.ipynb
    [FILE] 01_environment_and_dependencies.ipynb
    [F

## Step 5 – Implement the text processing module
Now we will move the preprocessing and chunking logic from Notebook 04 into a reusable Python module.

File to create:

src/processing/text_processor.py

**This module will:**
1. Normalize extracted PDF text.
2. Remove unnecessary whitespace and line-break hyphenation.
3. Split each page into overlapping chunks.
4. Preserve document ID, filename, page number, and chunk index.
5. Generate unique chunk IDs.
   
**We will keep the same settings used in Notebook 04:**
- Chunk size: 1200 characters
- Chunk overlap: 150 characters
- Page-wise chunking: enabled

**5.1 Create the Python module**

In [5]:

# Create the text processing module

from pathlib import Path

processor_file = (
    PROJECT_ROOT
    / "src"
    / "processing"
    / "text_processor.py"
)

processor_code = r'''
import re
import unicodedata
from pathlib import Path


# Default chunking configuration
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 150


def clean_text(text: str) -> str:
    """
    Normalize extracted scientific PDF text.

    Preserves original scientific terminology and symbols.
    """

    if not isinstance(text, str):
        raise TypeError("Input text must be a string.")

    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Remove soft hyphens
    text = text.replace("\u00ad", "")

    # Join words split by a hyphen at a line break
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

    # Normalize line breaks and whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def create_chunks(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP
) -> list:
    """
    Split text into overlapping character-based chunks.
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero.")

    if chunk_overlap < 0 or chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap must be non-negative and smaller than chunk_size."
        )

    text = clean_text(text)

    if not text:
        return []

    chunks = []
    start = 0

    while start < len(text):

        end = min(start + chunk_size, len(text))

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end == len(text):
            break

        start = end - chunk_overlap

    return chunks


def process_document(
    extracted_document: dict,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP
) -> list:
    """
    Convert an extracted PDF document into page-level chunks.

    Each chunk preserves its original document and page information.
    """

    filename = extracted_document["filename"]

    document_id = Path(filename).stem

    processed_chunks = []

    for page in extracted_document["pages"]:

        page_number = page["page_number"]

        page_chunks = create_chunks(
            page["text"],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        for chunk_index, chunk_text in enumerate(page_chunks, start=1):

            chunk_id = (
                f"{document_id}_p{page_number}_c{chunk_index}"
            )

            processed_chunks.append({
                "chunk_id": chunk_id,
                "document_id": document_id,
                "filename": filename,
                "page_number": page_number,
                "chunk_index": chunk_index,
                "text": chunk_text,
                "character_count": len(chunk_text),
                "word_count": len(chunk_text.split())
            })

    return processed_chunks
'''

processor_file.write_text(processor_code.strip() + "\n", encoding="utf-8")

print("Created:", processor_file)


Created: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant/src/processing/text_processor.py


**5.2 Test the module**

In [3]:

# Import the new processing functions

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.processing.text_processor import (
    clean_text,
    create_chunks,
    process_document
)

# Test text cleaning
sample_text = "Deep learn-\ning models   improve scientific analysis."

cleaned = clean_text(sample_text)

print("Cleaned text:")
print(cleaned)


# Test chunk creation
sample_chunks = create_chunks(
    "This is a sample scientific document. " * 100
)

print("\nNumber of test chunks:", len(sample_chunks))
print("First chunk length:", len(sample_chunks[0]))


# Test document processing using an existing PDF
from src.ingestion.pdf_extractor import extract_pdf

sample_pdf = (
    PROJECT_ROOT
    / "data"
    / "raw_pdfs"
    / "41598_2019_Article_38798.pdf"
)

extracted_document = extract_pdf(sample_pdf)

processed_chunks = process_document(extracted_document)

print("\nDocument processing results")
print("Filename:", extracted_document["filename"])
print("Pages:", extracted_document["page_count"])
print("Generated chunks:", len(processed_chunks))

print("\nFirst chunk:")
print(processed_chunks[0])


Cleaned text:
Deep learning models improve scientific analysis.

Number of test chunks: 4
First chunk length: 1200

Document processing results
Filename: 41598_2019_Article_38798.pdf
Pages: 14
Generated chunks: 60

First chunk:
{'chunk_id': '41598_2019_Article_38798_p1_c1', 'document_id': '41598_2019_Article_38798', 'filename': '41598_2019_Article_38798.pdf', 'page_number': 1, 'chunk_index': 1, 'text': 'www.nature.com/scientificreports OPEN Cell Identity Codes: Understanding Cell Identity from Gene Expression Profiles using Deep Neural Received: 5 September 2017Accepted: 10 January 2019 Networks Published: xx xx xxxx Farzad Abdolhosseini1, Behrooz Azarkhalili2, Abbas Maazallahi1, Aryan Kamal1, Seyed Abolfazl Motahari1, Ali Sharifi-Zarchi 1 & Hamidreza Chitsaz3 Understanding cell identity is an important task in many biomedical areas. Expression patterns of specific marker genes have been used to characterize some limited cell types, but exclusive markers are not available for many cell

In [4]:

import json

chunks_file = PROJECT_ROOT / "data" / "processed" / "document_chunks.jsonl"

with chunks_file.open("r", encoding="utf-8") as file:
    existing_chunks = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

print("Existing dataset:", len(existing_chunks), "chunks")

print("\nFirst three existing chunk IDs:")
for chunk in existing_chunks[:3]:
    print(chunk["chunk_id"])

print("\nNew processor chunk IDs:")
for chunk in processed_chunks[:3]:
    print(chunk["chunk_id"])


Existing dataset: 156 chunks

First three existing chunk IDs:
41598_2019_Article_38798_p1_c1
41598_2019_Article_38798_p1_c2
41598_2019_Article_38798_p1_c3

New processor chunk IDs:
41598_2019_Article_38798_p1_c1
41598_2019_Article_38798_p1_c2
41598_2019_Article_38798_p1_c3


## Step 6 – Embedding Model Integration
Now we will move the embedding functionality from Notebook 05 into a reusable Python module.

File to create:

src/embeddings/embedding_model.py

We will reuse your existing model and configuration:
Setting	Value
Embedding model	sentence-transformers/all-MiniLM-L6-v2
Embedding dimensions	384
Vector database	ChromaDB
Collection	research_papers
Database directory	data/chroma_db/
Similarity metric	Cosine

**Important:** We will connect to your existing ChromaDB database. We will not delete, recreate, or rebuild the existing 156 embeddings.

**6.1 Create the embedding module**

In [12]:

# Create the embedding model module

embedding_file = (
    PROJECT_ROOT
    / "src"
    / "embeddings"
    / "embedding_model.py"
)

embedding_code = '''
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer


class EmbeddingModel:
    """
    Generate text embeddings using Sentence Transformers.
    """

    def __init__(
        self,
        model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
        device: str | None = None
    ):
        self.model_name = model_name

        # Load the model on the selected device
        self.model = SentenceTransformer(
            model_name,
            device=device
        )

        self.embedding_dimension = self.model.get_embedding_dimension()

    def encode(self, texts: list[str]) -> list[list[float]]:
        """
        Convert a list of texts into normalized embeddings.
        """

        if not texts:
            return []

        embeddings = self.model.encode(
            texts,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        )

        return embeddings.tolist()


class ChromaVectorStore:
    """
    Manage persistent vector storage using ChromaDB.
    """

    def __init__(
        self,
        persist_directory: str | Path,
        collection_name: str = "research_papers"
    ):
        self.persist_directory = Path(persist_directory)
        self.persist_directory.mkdir(parents=True, exist_ok=True)

        # Connect to the existing persistent database
        self.client = chromadb.PersistentClient(
            path=str(self.persist_directory)
        )

        # Reuse the existing collection if it exists
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )

    def add_chunks(
        self,
        chunks: list[dict],
        embeddings: list[list[float]],
        batch_size: int = 100
    ) -> int:
        """
        Insert or update chunks and their embeddings.

        ChromaDB upsert prevents duplicate IDs.
        """

        if len(chunks) != len(embeddings):
            raise ValueError(
                "Number of chunks and embeddings must match."
            )

        if not chunks:
            return 0

        for start in range(0, len(chunks), batch_size):

            batch_chunks = chunks[start:start + batch_size]
            batch_embeddings = embeddings[start:start + batch_size]

            self.collection.upsert(
                ids=[chunk["chunk_id"] for chunk in batch_chunks],

                embeddings=batch_embeddings,

                documents=[
                    chunk["text"] for chunk in batch_chunks
                ],

                metadatas=[
                    {
                        "document_id": chunk["document_id"],
                        "filename": chunk["filename"],
                        "page_number": chunk["page_number"],
                        "chunk_index": chunk["chunk_index"],
                        "word_count": chunk["word_count"]
                    }
                    for chunk in batch_chunks
                ]
            )

        return len(chunks)

    def count(self) -> int:
        """Return the number of stored vectors."""
        return self.collection.count()

    def search(
        self,
        query_embedding: list[float],
        top_k: int = 5
    ) -> dict:
        """Search for semantically similar passages."""

        return self.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k,
            include=["documents", "metadatas", "distances"]
        )
'''

embedding_file.write_text(
    embedding_code.strip() + "\n",
    encoding="utf-8"
)

print("Created:", embedding_file)


Created: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant/src/embeddings/embedding_model.py


**6.2 Test the embedding model**

In [13]:

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.embeddings.embedding_model import (
    EmbeddingModel,
    ChromaVectorStore
)

# Initialize the embedding model
embedding_model = EmbeddingModel()

print("Embedding model:", embedding_model.model_name)
print("Embedding dimensions:", embedding_model.embedding_dimension)

# Generate a test embedding
test_text = "Deep learning models improve scientific research."

test_embedding = embedding_model.encode([test_text])

print("\nEmbedding shape:", len(test_embedding), "x", len(test_embedding[0]))
print("First five values:", test_embedding[0][:5])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimensions: 384

Embedding shape: 1 x 384
First five values: [-0.024926815181970596, -0.011452792212367058, 0.06010156869888306, 0.07170382887125015, 0.006076133344322443]


/home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant/src/embeddings/embedding_model.py:25: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.embedding_dimension = self.model.get_sentence_embedding_dimension()


**6.3 Connect to the existing ChromaDB database**

In [14]:

# Connect to the existing ChromaDB collection

CHROMA_PATH = PROJECT_ROOT / "data" / "chroma_db"

vector_store = ChromaVectorStore(
    persist_directory=CHROMA_PATH,
    collection_name="research_papers"
)

print("ChromaDB connection successful.")
print("Collection: research_papers")
print("Stored vectors:", vector_store.count())


ChromaDB connection successful.
Collection: research_papers
Stored vectors: 156


**6.4 Test semantic search**

In [15]:

# Generate a query embedding
query = "How do deep neural networks identify different cell types?"

query_embedding = embedding_model.encode([query])[0]

# Search the existing vector database
search_results = vector_store.search(
    query_embedding=query_embedding,
    top_k=3
)

# Display retrieved passages
for index in range(len(search_results["ids"][0])):

    print(f"\nResult {index + 1}")
    print("-" * 60)

    print("Chunk ID:", search_results["ids"][0][index])
    print("Filename:", search_results["metadatas"][0][index]["filename"])
    print("Page:", search_results["metadatas"][0][index]["page_number"])
    print("Cosine distance:", search_results["distances"][0][index])
    print("Text:", search_results["documents"][0][index][:300])



Result 1
------------------------------------------------------------
Chunk ID: 41598_2019_Article_38798_p5_c6
Filename: 41598_2019_Article_38798.pdf
Page: 5
Cosine distance: 0.40127986669540405
Text: cell type, while other CICs of the same sample could not be easily classified. In order to guide the training process of autoencoder to identify those neural network parameters that could produce more easily classifiable cell identity codes, we incorporated some additional layers to the network arch

Result 2
------------------------------------------------------------
Chunk ID: 41598_2019_Article_38798_p1_c1
Filename: 41598_2019_Article_38798.pdf
Page: 1
Cosine distance: 0.40380287170410156
Text: www.nature.com/scientificreports OPEN Cell Identity Codes: Understanding Cell Identity from Gene Expression Profiles using Deep Neural Received: 5 September 2017Accepted: 10 January 2019 Networks Published: xx xx xxxx Farzad Abdolhosseini1, Behrooz Azarkhalili2, Abbas Maazallahi1, Aryan Kamal1,

## Step 7 – Hybrid Retrieval and Reranking Module
File to create: src/retrieval/hybrid_retriever.py

We will move the proven retrieval logic from Notebook 06 into this module:
1. BM25 keyword retrieval.
2. ChromaDB semantic retrieval.
3. Reciprocal Rank Fusion (RRF).
4. Cross-Encoder reranking.
5. Return the top five passages with their metadata and scores.
Before implementing it, I want to reuse exact working Notebook 06 code rather than introduce differences in retrieval behavior.

**7.1 – Inspect the retrieval functions from Notebook 06**


In [16]:

import nbformat

notebook_path = (
    PROJECT_ROOT
    / "notebooks"
    / "06_hybrid_retrieval_and_reranking.ipynb"
)

notebook = nbformat.read(notebook_path, as_version=4)

keywords = [
    "BM25Okapi",
    "CrossEncoder",
    "def hybrid_retrieve",
    "def reciprocal_rank_fusion",
    "def semantic_search",
]

for cell_number, cell in enumerate(notebook.cells):

    if cell.cell_type != "code":
        continue

    source = cell.source

    if any(keyword in source for keyword in keywords):
        print("\n" + "=" * 70)
        print(f"CELL {cell_number}")
        print("=" * 70)
        print(source)



CELL 2
# Imports and configuration
import json
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import chromadb
import torch

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

CELL 7
# Initialize BM25 keyword retrieval
def tokenize(text: str) -> list[str]:
    """Convert text into lowercase word tokens."""
    return re.findall(r"\b\w+\b", text.lower())


tokenized_corpus = [
    tokenize(chunk["text"])
    for chunk in chunks
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index created.")
print("Indexed passages:", len(tokenized_corpus))

CELL 14
def semantic_search(
    query: str,
    top_k: int = SEMANTIC_TOP_K
) -> list:
    """Retrieve passages using ChromaDB semantic similarity."""

    query_embedding = embedding_model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    results = collection.query(
        query_embe

**Step 7.2 – Inspect the remaining retrieval functions**
Our previous output did not include three essential parts:
- bm25_search()
- rerank_results()
- Retrieval configuration constants.

In [17]:

# Inspect the remaining retrieval implementation

keywords = [
    "BM25_TOP_K",
    "SEMANTIC_TOP_K",
    "RRF_K",
    "RRF_TOP_K",
    "FINAL_TOP_K",
    "RERANKER_MODEL_NAME",
    "def bm25_search",
    "def rerank_results",
]

for cell_number, cell in enumerate(notebook.cells):

    if cell.cell_type != "code":
        continue

    source = cell.source

    if any(keyword in source for keyword in keywords):
        print("\n" + "=" * 70)
        print(f"CELL {cell_number}")
        print("=" * 70)
        print(source)



CELL 3
# Set up project paths and configuration:
PROJECT_ROOT = (
    Path.home()
    / "Schreibtisch"
    / "IfaDW"
    / "NLP"
    / "Final_Project"
    / "advance-ai-research-assistant"
)

CHUNKS_FILE = PROJECT_ROOT / "data" / "processed" / "document_chunks.jsonl"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma_db"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

SEMANTIC_TOP_K = 10
BM25_TOP_K = 10
RRF_TOP_K = 10
FINAL_TOP_K = 5

RRF_K = 60

print("Project:", PROJECT_ROOT)
print("Chunks file exists:", CHUNKS_FILE.exists())
print("ChromaDB exists:", CHROMA_DIR.exists())
print("CUDA available:", torch.cuda.is_available())

assert CHUNKS_FILE.exists()
assert CHROMA_DIR.exists()

CELL 8
# Creating a BM25 search function:
def bm25_search(query: str, top_k: int = BM25_TOP_K) -> list:
    """Retrieve passages using BM25 

**Step 7.3 – Create hybrid_retriever.py**
Creating the file:

src/retrieval/hybrid_retriever.py

This implementation uses your existing EmbeddingModel and ChromaVectorStore classes. It also supports dependency injection, so we can reuse the objects already loaded in Notebook 08 instead of unnecessarily loading the embedding model again.

In [18]:

# Create the hybrid retrieval module

retriever_file = (
    PROJECT_ROOT
    / "src"
    / "retrieval"
    / "hybrid_retriever.py"
)

retriever_code = r'''
import json
import re
from pathlib import Path
from collections import defaultdict

from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

from src.embeddings.embedding_model import (
    EmbeddingModel,
    ChromaVectorStore
)


# Project configuration
PROJECT_ROOT = Path(__file__).resolve().parents[2]

CHUNKS_FILE = PROJECT_ROOT / "data" / "processed" / "document_chunks.jsonl"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma_db"

RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

BM25_TOP_K = 10
SEMANTIC_TOP_K = 10
RRF_TOP_K = 10
FINAL_TOP_K = 5
RRF_K = 60


class HybridRetriever:
    """
    Combine BM25, semantic retrieval, RRF fusion,
    and Cross-Encoder reranking.
    """

    def __init__(
        self,
        embedding_model=None,
        vector_store=None,
        reranker=None,
        chunks_file: str | Path = CHUNKS_FILE,
        reranker_model_name: str = RERANKER_MODEL_NAME
    ):

        self.chunks_file = Path(chunks_file)

        if not self.chunks_file.exists():
            raise FileNotFoundError(
                f"Chunk file not found: {self.chunks_file}"
            )

        # Load the existing document chunks
        with self.chunks_file.open("r", encoding="utf-8") as file:
            self.chunks = [
                json.loads(line)
                for line in file
                if line.strip()
            ]

        if not self.chunks:
            raise ValueError("No document chunks were found.")

        # Initialize or reuse the embedding model
        self.embedding_model = (
            embedding_model
            if embedding_model is not None
            else EmbeddingModel()
        )

        # Initialize or reuse the ChromaDB connection
        self.vector_store = (
            vector_store
            if vector_store is not None
            else ChromaVectorStore(CHROMA_DIR)
        )

        # Create the BM25 index
        self.tokenized_corpus = [
            self.tokenize(chunk["text"])
            for chunk in self.chunks
        ]

        self.bm25 = BM25Okapi(self.tokenized_corpus)

        # Initialize or reuse the Cross-Encoder
        self.reranker = (
            reranker
            if reranker is not None
            else CrossEncoder(
                reranker_model_name,
                device="cpu",
                max_length=512
            )
        )

    @staticmethod
    def tokenize(text: str) -> list[str]:
        """Convert text into lowercase word tokens."""
        return re.findall(r"\b\w+\b", text.lower())

    def bm25_search(
        self,
        query: str,
        top_k: int = BM25_TOP_K
    ) -> list:
        """Retrieve passages using BM25 keyword matching."""

        query_tokens = self.tokenize(query)

        scores = self.bm25.get_scores(query_tokens)

        ranked_indices = scores.argsort()[::-1][:top_k]

        results = []

        for index in ranked_indices:

            chunk = self.chunks[index]

            results.append({
                "chunk_id": chunk["chunk_id"],
                "score": float(scores[index]),
                "text": chunk["text"],
                "metadata": {
                    "filename": chunk["filename"],
                    "page_number": chunk["page_number"],
                    "document_id": chunk["document_id"],
                    "chunk_index": chunk["chunk_index"]
                }
            })

        return results

    def semantic_search(
        self,
        query: str,
        top_k: int = SEMANTIC_TOP_K
    ) -> list:
        """Retrieve passages using ChromaDB semantic similarity."""

        collection_count = self.vector_store.count()

        if collection_count == 0:
            return []

        query_embedding = self.embedding_model.encode([query])[0]

        results = self.vector_store.search(
            query_embedding=query_embedding,
            top_k=min(top_k, collection_count)
        )

        retrieved = []

        if not results["ids"] or not results["ids"][0]:
            return retrieved

        for index, chunk_id in enumerate(results["ids"][0]):

            retrieved.append({
                "chunk_id": chunk_id,
                "score": float(results["distances"][0][index]),
                "text": results["documents"][0][index],
                "metadata": results["metadatas"][0][index]
            })

        return retrieved

    @staticmethod
    def reciprocal_rank_fusion(
        result_lists: list,
        k: int = RRF_K,
        top_k: int = RRF_TOP_K
    ) -> list:
        """Combine multiple rankings using Reciprocal Rank Fusion."""

        rrf_scores = defaultdict(float)
        results_by_id = {}

        for results in result_lists:

            for rank, result in enumerate(results, start=1):

                chunk_id = result["chunk_id"]

                rrf_scores[chunk_id] += 1.0 / (k + rank)

                results_by_id[chunk_id] = result

        ranked_ids = sorted(
            rrf_scores,
            key=lambda chunk_id: rrf_scores[chunk_id],
            reverse=True
        )

        fused_results = []

        for chunk_id in ranked_ids[:top_k]:

            result = results_by_id[chunk_id].copy()

            result["rrf_score"] = rrf_scores[chunk_id]

            fused_results.append(result)

        return fused_results

    def rerank_results(
        self,
        query: str,
        results: list,
        top_k: int = FINAL_TOP_K
    ) -> list:
        """Rerank passages using the Cross-Encoder."""

        if not results:
            return []

        pairs = [
            (query, result["text"])
            for result in results
        ]

        scores = self.reranker.predict(
            pairs,
            show_progress_bar=False
        )

        reranked = []

        for result, score in zip(results, scores):

            item = result.copy()
            item["rerank_score"] = float(score)

            reranked.append(item)

        reranked.sort(
            key=lambda item: item["rerank_score"],
            reverse=True
        )

        return reranked[:top_k]

    def hybrid_retrieve(
        self,
        query: str,
        final_top_k: int = FINAL_TOP_K
    ) -> list:
        """Execute the complete hybrid retrieval pipeline."""

        if not query or not query.strip():
            raise ValueError("The search query cannot be empty.")

        # Step 1: BM25 keyword retrieval
        keyword_results = self.bm25_search(
            query,
            top_k=BM25_TOP_K
        )

        # Step 2: Semantic retrieval
        vector_results = self.semantic_search(
            query,
            top_k=SEMANTIC_TOP_K
        )

        # Step 3: Reciprocal Rank Fusion
        fused_results = self.reciprocal_rank_fusion(
            [keyword_results, vector_results],
            top_k=RRF_TOP_K
        )

        # Step 4: Cross-Encoder reranking
        final_results = self.rerank_results(
            query,
            fused_results,
            top_k=final_top_k
        )

        return final_results
'''

retriever_file.write_text(
    retriever_code.strip() + "\n",
    encoding="utf-8"
)

print("Created:", retriever_file)


Created: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant/src/retrieval/hybrid_retriever.py


**Step 7.4 – Test the hybrid retriever**
We will reuse the embedding model and ChromaDB connection from Step 6.

In [19]:

import importlib

import src.retrieval.hybrid_retriever as retrieval_module

importlib.reload(retrieval_module)

from src.retrieval.hybrid_retriever import HybridRetriever

# Initialize the retriever
hybrid_retriever = HybridRetriever(
    embedding_model=embedding_model,
    vector_store=vector_store
)

print("Hybrid retriever initialized.")
print("BM25 passages:", len(hybrid_retriever.chunks))
print("ChromaDB vectors:", hybrid_retriever.vector_store.count())


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Hybrid retriever initialized.
BM25 passages: 156
ChromaDB vectors: 156


**Step 7.5 – Test a research question**

In [20]:

# Test the complete retrieval pipeline

test_question = (
    "How do deep neural networks help identify different cell types?"
)

retrieved_passages = hybrid_retriever.hybrid_retrieve(
    test_question,
    final_top_k=5
)

print("Research question:", test_question)
print("\nRetrieved passages:", len(retrieved_passages))

for rank, result in enumerate(retrieved_passages, start=1):

    print("\n" + "=" * 70)
    print(f"RANK {rank}")
    print("=" * 70)

    print("Chunk ID:", result["chunk_id"])
    print("Filename:", result["metadata"]["filename"])
    print("Page:", result["metadata"]["page_number"])
    print("RRF score:", round(result["rrf_score"], 6))
    print("Cross-Encoder score:", round(result["rerank_score"], 4))
    print("Text:", result["text"][:350])


Research question: How do deep neural networks help identify different cell types?

Retrieved passages: 5

RANK 1
Chunk ID: 41598_2019_Article_38798_p1_c1
Filename: 41598_2019_Article_38798.pdf
Page: 1
RRF score: 0.032787
Cross-Encoder score: 3.3362
Text: www.nature.com/scientificreports OPEN Cell Identity Codes: Understanding Cell Identity from Gene Expression Profiles using Deep Neural Received: 5 September 2017Accepted: 10 January 2019 Networks Published: xx xx xxxx Farzad Abdolhosseini1, Behrooz Azarkhalili2, Abbas Maazallahi1, Aryan Kamal1, Seyed Abolfazl Motahari1, Ali Sharifi-Zarchi 1 & Hamid

RANK 2
Chunk ID: 41598_2019_Article_38798_p1_c2
Filename: 41598_2019_Article_38798.pdf
Page: 1
RRF score: 0.032258
Cross-Encoder score: 2.4999
Text: ific structure of deep autoencoders that can encode a GEP into a vector of 30 numeric values, which we call the cell identity code (CIC). The original GEP can be reproduced from the CIC with an accuracy comparable to technical replicates of th